# Lab 6 — Completed Exercises

**Student:** Abdulaziz Saqer BinRuqush  
**Student ID:** 2230007229

Instructor notebook: [Lab6_deep_learning_for_nlp-3.ipynb](https://github.com/MohammedMosuily/NLP_Lab6)  
Source commit: `88a67abcb855d7ccc7b83635f917f536ec5b02f6`

Run the setup commands in the course README before running all cells.
Instructor examples and task instructions are retained below.


In [1]:
from pathlib import Path
import os
import sys
import random
import numpy as np
import pandas as pd
from IPython.display import display

LAB_DIR = Path.cwd()
if not (LAB_DIR / 'Lab6_deep_learning_for_nlp-3.ipynb').is_file():
    LAB_DIR = LAB_DIR / "Lab6"
if not (LAB_DIR / 'Lab6_deep_learning_for_nlp-3.ipynb').is_file():
    raise FileNotFoundError("Run from the course root or this lab's folder.")
PROJECT_ROOT = LAB_DIR.parent
os.environ.setdefault("KAGGLEHUB_CACHE", str(PROJECT_ROOT / ".cache" / "kagglehub"))
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))
os.environ.setdefault("NLTK_DATA", str(PROJECT_ROOT / ".cache" / "nltk_data"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from course_data import dataset_file
import nltk
nltk.data.path.insert(0, os.environ["NLTK_DATA"])
random.seed(42)
np.random.seed(42)
print("Lab 6 | Abdulaziz Saqer BinRuqush | 2230007229")


Lab 6 | Abdulaziz Saqer BinRuqush | 2230007229


# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [2]:
df = pd.read_csv(LAB_DIR / "dataset" / "07-yelp-dataset.txt", sep="\t", names=["review", "label"])
print("Number of reviews:", len(df))
assert df["label"].isin([0, 1]).all()
display(df.head())


Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [4]:
import re
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()
df["review"] = df["review"].apply(preprocess_text)
display(df.head())


,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [5]:
from sklearn.model_selection import train_test_split
X = df["review"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))


Training reviews:

 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [7]:
import torch
import torch.nn as nn
torch.manual_seed(42)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)


## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [8]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [9]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [10]:
epochs = 10
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")
assert torch.isfinite(loss)


Epoch 5/10 - Loss: 0.6259
Epoch 10/10 - Loss: 0.3552


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.760


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [12]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"], zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))


Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.69      0.74       100
    Positive       0.73      0.83      0.78       100

    accuracy                           0.76       200
   macro avg       0.77      0.76      0.76       200
weighted avg       0.77      0.76      0.76       200

Confusion Matrix:
[[69 31]
 [17 83]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [13]:
from gensim.models import Word2Vec
train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]
word2vec = Word2Vec(sentences=train_tokens, vector_size=200, window=5,
                   min_count=1, sg=1, seed=42, workers=1)

def review_to_vector(tokens, word_vectors, vector_size):
    known_vectors = [word_vectors[token] for token in tokens if token in word_vectors]
    if not known_vectors:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(known_vectors, axis=0).astype(np.float32)

train_review_vectors = np.vstack([review_to_vector(tokens, word2vec.wv, 200) for tokens in train_tokens])
test_review_vectors = np.vstack([review_to_vector(tokens, word2vec.wv, 200) for tokens in test_tokens])
# Center and scale using training reviews only. The same transform is applied to test reviews.
from sklearn.preprocessing import StandardScaler
print("Median training embedding feature standard deviation:",
      float(np.median(train_review_vectors.std(axis=0))))
embedding_scaler = StandardScaler()
train_review_vectors = embedding_scaler.fit_transform(train_review_vectors)
test_review_vectors = embedding_scaler.transform(test_review_vectors)
assert embedding_scaler.n_samples_seen_ == len(X_train)
X_train_w2v_tensor = torch.tensor(train_review_vectors, dtype=torch.float32)
X_test_w2v_tensor = torch.tensor(test_review_vectors, dtype=torch.float32)
print("Word2Vec training shape:", X_train_w2v_tensor.shape)
print("Word2Vec testing shape:", X_test_w2v_tensor.shape)
print("Training vocabulary:", len(word2vec.wv))
assert X_train_w2v_tensor.shape == (len(X_train), 200)
assert X_test_w2v_tensor.shape == (len(X_test), 200)
assert torch.isfinite(X_train_w2v_tensor).all() and torch.isfinite(X_test_w2v_tensor).all()


Median training embedding feature standard deviation: 0.0077226972207427025
Word2Vec training shape: torch.Size([800, 200])
Word2Vec testing shape: torch.Size([200, 200])
Training vocabulary: 1787


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [14]:
torch.manual_seed(42)
word2vec_network = nn.Sequential(
    nn.Linear(200, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1))
print(word2vec_network)
w2v_loss_function = nn.BCEWithLogitsLoss()
w2v_optimizer = torch.optim.Adam(word2vec_network.parameters(), lr=0.001)
w2v_epochs = 15
losses = []
for epoch in range(w2v_epochs):
    word2vec_network.train()
    w2v_optimizer.zero_grad()
    outputs = word2vec_network(X_train_w2v_tensor)
    loss = w2v_loss_function(outputs, y_train_tensor)
    loss.backward()
    w2v_optimizer.step()
    losses.append(loss.item())
    print(f"Epoch {epoch + 1}/{w2v_epochs} - Loss: {loss.item():.4f}")
word2vec_network.eval()
with torch.no_grad():
    w2v_probabilities = torch.sigmoid(word2vec_network(X_test_w2v_tensor))
    w2v_predictions = (w2v_probabilities >= 0.5).to(torch.int64).numpy().flatten()
w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Word2Vec network test accuracy: {w2v_accuracy:.3f}")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, w2v_predictions))
assert len(losses) == 15 and np.isfinite(losses).all()
assert [layer.out_features for layer in word2vec_network if isinstance(layer, nn.Linear)] == [128, 64, 32, 1]
assert w2v_predictions.shape == (len(y_test),)


Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)
Epoch 1/15 - Loss: 0.6969
Epoch 2/15 - Loss: 0.6944
Epoch 3/15 - Loss: 0.6932
Epoch 4/15 - Loss: 0.6918
Epoch 5/15 - Loss: 0.6904
Epoch 6/15 - Loss: 0.6891
Epoch 7/15 - Loss: 0.6876
Epoch 8/15 - Loss: 0.6859
Epoch 9/15 - Loss: 0.6840
Epoch 10/15 - Loss: 0.6819
Epoch 11/15 - Loss: 0.6794
Epoch 12/15 - Loss: 0.6767
Epoch 13/15 - Loss: 0.6737
Epoch 14/15 - Loss: 0.6705
Epoch 15/15 - Loss: 0.6667
Word2Vec network test accuracy: 0.585
              precision    recall  f1-score   support

    Negative       0.72      0.28      0.40       100
    Positive       0.55      0.89      0.68       100

    accuracy                           0.58       200
   macro avg       0.64      0.58      0.54       200
weig

## Representation and training notes

Word2Vec sees only training reviews. Each review is represented by the mean
of its known token embeddings. Unknown test tokens are ignored; an empty or
entirely unknown review receives a 200-dimensional zero vector.

The required network has three hidden layers and outputs a raw logit.
BCEWithLogitsLoss combines sigmoid and binary cross-entropy during training;
evaluation applies sigmoid and the 0.5 threshold. Gradients are reset before
each update. Both models use the same held-out reviews for comparison.

The averaged embeddings have small feature variation around a shared mean. StandardScaler centers and scales each feature using only training reviews, then applies that fitted transformation to test reviews. This improves the numerical conditioning of the neural-network input while retaining 200 features and the required architecture, learning rate, and 15 training epochs.
